In [4]:
import pandas as pd
import numpy as np
import math

from open_dataset_store import quick_start

# Ensure the simulation files can be imported
import sys
sys.path.append("/home/jazz/Projects/DHCA-Framework/EnergyPlusSim")
from _5ZoneAutoDXVAV_zone_controller import ZoneController, get_w_from_rh
from _5ZoneAutoDXVAV_ahu_coordinator import AHUCoordinator


In [2]:
# Simple logger to collect data row-by-row
class SimpleLogger:
    def __init__(self):
        self.data = {}
    def add(self, key, value):
        self.data[key] = value

# 1. Initialize Dataset Store & Load Raw Data
store = quick_start('.', backend='local')

Store initialised at: . (Backend: local)


In [37]:
entry_id = "entry_0005" # Update this to your entry_id!

In [38]:

def calculate_actual_flow(C, M):
    """
    Calculates the actual air mass flow (kg/s) based on recorded fan and mixer commands.
    C = Fan Command percentage (0 to 90)
    M = Mixer Command (0 for closed, 100 for open)
    """
    if C <= 0:
        return 0.0
    # Equation: F = (9.3 + 0.008M) / (1 + exp(-0.16(C - 49 + 0.01M)))
    return (9.3 + 0.008 * M) / (1.0 + math.exp(-0.16 * (C - 49.0 + 0.01 * M)))

def calculate_fan_command(F, M):
    """
    Calculates the required Fan Command % to achieve a requested mass flow F.
    """
    if F <= 0:
        return 0.0
    val = (9.3 + 0.008 * M) / F - 1.0
    if val <= 0:
        return 90.0
    C = 49.0 - 0.01 * M - math.log(val) / 0.16
    return max(0.0, min(90.0, C))


df = store.get_entry_raw_data("experiments", entry_id)

# 2. Initialize Controllers
zone_controller = ZoneController("Zone_1")
ahu_coordinator = AHUCoordinator()

# Track previous AHU states to feed into next zone step
prev_ahu_T_s = 13.0
prev_ahu_W_s = 0.008
prev_ahu_C_s = 420.0

results = []
prev_time = None

print("Starting offline evaluation loop...")

# 3. Evaluation Loop
for i, row in df.iterrows():
    # Calculate dt dynamically
    curr_time_dt = pd.to_datetime(row['timestamp'])
    curr_time = curr_time_dt.timestamp()
    
    if prev_time is None:
        dt = 5.0
    else:
        dt = max(1.0, curr_time - prev_time)
    prev_time = curr_time
    
    step_logger = SimpleLogger()
    step_logger.add("timestamp", row['timestamp'])
    
    # Parse dataset row
    T_out = row['outside_t']
    RH_out = row['outside_h'] / 100.0 if row['outside_h'] > 1.0 else row['outside_h']
    W_out = get_w_from_rh(T_out, RH_out)
    C_out = row['outside_c']
    
    T_in = row['room_1_t']
    RH_in = row['room_1_h'] / 100.0 if row['room_1_h'] > 1.0 else row['room_1_h']
    W_in = get_w_from_rh(T_in, RH_in)
    C_in = row['room_2_c']
    
    # ⚠️ Calculate real flowrate from recorded commands to feed into EKF
    recorded_fan = row['fan']
    recorded_mixer = row['mixer']
    vav_flow_real = calculate_actual_flow(recorded_fan, recorded_mixer)
    
    # Construct state for ZoneController
    state_data = {
        'T_out': T_out,
        'W_out': W_out,
        'C_out': C_out,
        'T_in': T_in,
        'W_in': W_in,
        'C_in': C_in,
        'T_s': prev_ahu_T_s,
        'W_s': prev_ahu_W_s,
        'C_s': prev_ahu_C_s,
        'temp_setpoint': 22.0, 
        'VAV_Flow': vav_flow_real
    }
    
    # Step ZoneController (Returns a Dictionary!)
    zone_res = zone_controller.step(dt, state_data, step_logger)
    u_cmd = zone_res.get('u_cmd', 0.0)
    
    # Step AHU Coordinator
    zone_conditions = {'Zone_1': zone_res}
    system_state = {
        'T_out': T_out,
        'C_out': C_out,
        'T_ret': T_in,
        'C_ret': C_in,
        'actual_co2': C_in,
        'fan_out_T': prev_ahu_T_s,
        'hc_out_T': prev_ahu_T_s # Assuming no extra heating at HC for basic offline sim
    }
    
    ahu_res = ahu_coordinator.step(zone_conditions, system_state, curr_time, step_logger)
    
    # Calculate the MPC Actuation Signals 
    oa_flow_sp = ahu_res.get('oa_flow_sp', 0.0)
    gamma = oa_flow_sp / 2.5
    mpc_mixer_ratio = max(0.0, min(100.0, gamma * 100.0))
    mpc_fan_cmd = calculate_fan_command(u_cmd, mpc_mixer_ratio)
    
    # Update AHU states for the next iteration
    prev_ahu_T_s = ahu_res.get('cc_temp_sp', 13.0)
    prev_ahu_W_s = ahu_res.get('hum_w_sp', 0.008)
    prev_ahu_C_s = C_out 
    
    # Log MPC actions & commands
    step_logger.add("Actual_Calculated_Flow", vav_flow_real)
    step_logger.add("MPC_u_cmd_flow", u_cmd)
    step_logger.add("MPC_Fan_Cmd", mpc_fan_cmd)
    step_logger.add("MPC_Mixer_Ratio", mpc_mixer_ratio)
    
    # Log EKF Internal States
    ekf_states = ["EKF_T_in", "EKF_T_m", "EKF_W_in", "EKF_C_in", "EKF_d_T", "EKF_d_W", 
                  "EKF_N_occ", "EKF_alpha_ext", "EKF_alpha_int", "EKF_beta_air", "EKF_beta_mass", "EKF_d_C"]
    for idx, state_val in enumerate(zone_controller.x):
        step_logger.add(ekf_states[idx], state_val)
        
    results.append(step_logger.data)

print("Evaluation loop complete. Saving processed data...")

# 4. Save Processed Data to Store
results_df = pd.DataFrame(results)

store.add_processed_data(
    entry_type="experiments",
    entry_id=entry_id,  
    tag="controller_evaluation",
    df=results_df,
    description="Offline simulation of EKF/MPC and AHU Coordinator against dataset"
)

print(f"✅ Controller evaluation successfully added to entry {entry_id} with tag 'controller_evaluation'!")


Starting offline evaluation loop...
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zone_1] OSQP solved, status=1
[Zo

In [39]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd


# Fetch the raw and processed datasets
df_raw = store.get_entry_raw_data("experiments", entry_id)
results_df = store.get_entry_processed_data("experiments", entry_id, "controller_evaluation")

# ==========================================
# 2. Data Preparation
# ==========================================
def rh_to_w(T, rh):
    rh_frac = rh / 100.0 if (rh > 1.0).any() else rh
    p_sat = 610.94 * np.exp(17.625 * T / (243.04 + T))
    p_vapor = rh_frac * p_sat
    return 0.62198 * p_vapor / (101325.0 - p_vapor)

# We use the timestamp from the results_df which has already been localized
time_arr = pd.to_datetime(results_df['timestamp'], format='mixed')
raw_w_in = rh_to_w(df_raw['room_1_t'], df_raw['room_1_h'])

# ==========================================
# 3. Plotting
# ==========================================
fig = make_subplots(
    rows=7, cols=1,
    subplot_titles=[
        "Temperature Estimation (T_in vs EKF_T_in & EKF_T_m)",
        "Humidity Estimation (W_in vs EKF_W_in)",
        "CO2 Estimation (C_in vs EKF_C_in)",
        "Occupancy Estimation (EKF_N_occ vs Actual)",
        "Disturbances (d_T, d_W, d_C)",
        "Model Parameters (Alphas & Betas)",
        "Actuation: Air Flow & Economizer Mixer"
    ],
    vertical_spacing=0.04,
    shared_xaxes=True,
    specs=[
        [{"secondary_y": False}], # 1. Temp
        [{"secondary_y": False}], # 2. Hum
        [{"secondary_y": False}], # 3. CO2
        [{"secondary_y": False}], # 4. Occ
        [{"secondary_y": True}],  # 5. Disturbances 
        [{"secondary_y": True}],  # 6. Params
        [{"secondary_y": True}]   # 7. Flow & Commands (Secondary Y enabled)

    ]
)

# --- 1. Temperature (T, EKF_T, EKF_Tm) ---
fig.add_trace(go.Scatter(x=time_arr, y=df_raw['room_1_t'], name="Raw T_in", line=dict(color="rgba(255,255,255,0.3)", width=3)), row=1, col=1)
fig.add_trace(go.Scatter(x=time_arr, y=results_df['EKF_T_in'], name="EKF T_in", line=dict(color="#2ecc71", width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=time_arr, y=results_df['EKF_T_m'], name="EKF T_m (Wall Mass)", line=dict(color="#f1c40f", width=2, dash='dot')), row=1, col=1)

# --- 2. Humidity (W vs EKF_W) ---
fig.add_trace(go.Scatter(x=time_arr, y=raw_w_in, name="Raw W_in", line=dict(color="rgba(255,255,255,0.3)", width=3)), row=2, col=1)
fig.add_trace(go.Scatter(x=time_arr, y=results_df['EKF_W_in'], name="EKF W_in", line=dict(color="#3498db", width=2)), row=2, col=1)

# --- 3. CO2 (C vs EKF_C) ---
fig.add_trace(go.Scatter(x=time_arr, y=df_raw['room_2_c'], name="Raw C_in", line=dict(color="rgba(255,255,255,0.3)", width=3)), row=3, col=1)
fig.add_trace(go.Scatter(x=time_arr, y=results_df['EKF_C_in'], name="EKF C_in", line=dict(color="#e67e22", width=2)), row=3, col=1)

# --- 4. Occupancy (Actual vs EKF_N_occ) ---
# Assuming 'actual_occupancy' is now permanently part of df_raw after your previous update
if 'actual_occupancy' in df_raw.columns:
    fig.add_trace(go.Scatter(x=time_arr, y=df_raw['actual_occupancy'], name="Actual Occupancy", line=dict(color="rgba(255,255,255,0.6)", width=3, shape='hv')), row=4, col=1)
fig.add_trace(go.Scatter(x=time_arr, y=results_df['EKF_N_occ'], name="EKF N_occ", line=dict(color="#9b59b6", width=2), fill='tozeroy', fillcolor="rgba(155, 89, 182, 0.2)"), row=4, col=1)

# --- 5. Disturbances (d_T, d_W, d_C) ---
fig.add_trace(go.Scatter(x=time_arr, y=results_df['EKF_d_T'], name="d_T (Heat Disturbance)", line=dict(color="#e74c3c", width=1.5)), row=5, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=time_arr, y=results_df['EKF_d_C'], name="d_C (CO2 Disturbance)", line=dict(color="#e67e22", width=1.5, dash='dash')), row=5, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=time_arr, y=results_df['EKF_d_W'], name="d_W (Latent Disturbance)", line=dict(color="#3498db", width=1.5)), row=5, col=1, secondary_y=True)

# --- 6. Parameters (Alphas & Betas) ---
fig.add_trace(go.Scatter(x=time_arr, y=results_df['EKF_alpha_int'], name="Alpha_int", line=dict(color="#1abc9c", width=1.5)), row=6, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=time_arr, y=results_df['EKF_alpha_ext'], name="Alpha_ext", line=dict(color="#16a085", width=1.5, dash='dash')), row=6, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=time_arr, y=results_df['EKF_beta_air'], name="Beta_air", line=dict(color="#f39c12", width=1.5)), row=6, col=1, secondary_y=True)
fig.add_trace(go.Scatter(x=time_arr, y=results_df['EKF_beta_mass'], name="Beta_mass", line=dict(color="#d35400", width=1.5, dash='dash')), row=6, col=1, secondary_y=True)




# --- 7. Flow & Commands (Primary: kg/s, Secondary: %) ---
# Flows (Primary Y Axis)
fig.add_trace(go.Scatter(x=time_arr, y=results_df['Actual_Calculated_Flow'], name="Actual Calc Flow (kg/s)", line=dict(color="rgba(46, 204, 113, 0.4)", width=3)), row=7, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=time_arr, y=results_df['MPC_u_cmd_flow'], name="MPC Command Flow (kg/s)", line=dict(color="#2ecc71", width=2, dash='dot')), row=7, col=1, secondary_y=False)
# Fan & Mixer Percentages (Secondary Y Axis)
fig.add_trace(go.Scatter(x=time_arr, y=df_raw['fan'], name="Raw Fan Cmd (%)", line=dict(color="rgba(231, 76, 60, 0.4)", width=3)), row=7, col=1, secondary_y=True)
fig.add_trace(go.Scatter(x=time_arr, y=results_df['MPC_Fan_Cmd'], name="MPC Fan Cmd (%)", line=dict(color="#e74c3c", width=2, dash='dot')), row=7, col=1, secondary_y=True)
fig.add_trace(go.Scatter(x=time_arr, y=df_raw['mixer'], name="Raw Mixer (%)", line=dict(color="rgba(52, 152, 219, 0.4)", width=3)), row=7, col=1, secondary_y=True)
fig.add_trace(go.Scatter(x=time_arr, y=results_df['MPC_Mixer_Ratio'], name="MPC Mixer (%)", line=dict(color="#3498db", width=2, dash='dot')), row=7, col=1, secondary_y=True)


# --- Layout Styling ---
fig.update_layout(
    height=1900, 
    template="plotly_dark",
    title_text=f"Offline EKF Estimation Diagnostics ({entry_id})",
    margin=dict(l=30, r=30, t=80, b=30),
    hovermode="x unified"
)

# Axis titles
fig.update_yaxes(title_text="Temp (°C)", row=1, col=1)
fig.update_yaxes(title_text="Abs Hum (kg/kg)", row=2, col=1)
fig.update_yaxes(title_text="CO2 (ppm)", row=3, col=1)
fig.update_yaxes(title_text="People", row=4, col=1)

fig.update_yaxes(title_text="d_T / d_C", row=5, col=1, secondary_y=False)
fig.update_yaxes(title_text="d_W", row=5, col=1, secondary_y=True, showgrid=False)

fig.update_yaxes(title_text="Alphas (Conductance)", row=6, col=1, secondary_y=False)
fig.update_yaxes(title_text="Betas (1/Capacitance)", row=6, col=1, secondary_y=True, showgrid=False)

fig.update_yaxes(title_text="Air Flow (kg/s)", row=7, col=1, secondary_y=False)
fig.update_yaxes(title_text="Command (%)", range=[0, 100], row=7, col=1, secondary_y=True, showgrid=False)


fig.show()
